<a href='https://colab.research.google.com/github/Emelecto/QuantLab/blob/main/web/content/cursos/python/notebooks/c2_l5.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg'/></a>

# C2-L5 · Cliente Binance klines (demo)
Descargamos velas reales de Binance y las normalizamos. Con fallback al CSV si no hay red.

In [ ]:
import pandas as pd
from pathlib import Path

URL_CSV = 'https://raw.githubusercontent.com/Emelecto/QuantLab/main/web/content/cursos/python/data/c2_l5.csv'
COLS = ['open_time', 'open', 'high', 'low', 'close', 'volume']

def get_klines(symbol='BTCUSDT', interval='1d', limit=50):
    import requests
    url = 'https://api.binance.com/api/v3/klines'
    try:
        r = requests.get(url, params={'symbol': symbol, 'interval': interval, 'limit': limit}, timeout=10)
        r.raise_for_status()
        raw = r.json()
        df = pd.DataFrame(raw)[[0, 1, 2, 3, 4, 5]]
        df.columns = COLS
        df['open_time'] = pd.to_datetime(df['open_time'], unit='ms', utc=True)
        for c in ['open', 'high', 'low', 'close', 'volume']:
            df[c] = df[c].astype(float)
        print(f'Fuente: Binance API ({symbol} {interval})')
    except Exception as e:
        print('Sin red, uso fallback local:', e)
        for cand in [URL_CSV, Path('../data/c2_l5.csv'), Path('data/c2_l5.csv'), Path('c2_l5.csv')]:
            try:
                df = pd.read_csv(cand, parse_dates=['open_time'])
                break
            except Exception:
                continue
        print('Fuente: CSV demo')
    return df.sort_values('open_time').reset_index(drop=True)

df = get_klines()
print(df.shape)
print(df.head(5).to_string(index=False))

In [ ]:
df['ret'] = df['close'].pct_change()
df['rango'] = (df['high'] - df['low']) / df['close']
print(df[['open_time', 'close', 'ret', 'rango', 'volume']].tail(8).to_string(index=False))
print(f'\nret_medio={df["ret"].mean():.4%}  vol_media={df["volume"].mean():.1f}')

In [ ]:
# Chequeo automático
assert list(df.columns[:6]) == ['open_time', 'open', 'high', 'low', 'close', 'volume']
assert pd.api.types.is_float_dtype(df['close']), 'close debe ser float'
assert pd.api.types.is_datetime64_any_dtype(df['open_time']), 'open_time debe ser datetime'
assert (df['high'] >= df[['open', 'close']].max(axis=1) - 1e-9).all(), 'high >= open,close'
assert (df['low'] <= df[['open', 'close']].min(axis=1) + 1e-9).all(), 'low <= open,close'
assert df['open_time'].is_monotonic_increasing, 'orden cronológico'
print('OK: klines verificados,', len(df), 'velas')